In [1]:
# 7. Optimasi

import pandas as pd
import pulp
from geopy.distance import geodesic

# 7.1. Memuat Data Prediksi & Koordinat
df_pred = pd.read_csv('../data/processed/prediksi_masa_depan.csv')
df_spatial = pd.read_csv('../data/raw/koordinat_provinsi.csv')
df = pd.merge(df_pred, df_spatial, on='Province_Name')

# 7.2. Menentukan Provinsi Surplus dan Defisit
# Mengurutkan berdasarkan selisih harga (Price_Difference)
df = df.sort_values('Price_Difference').reset_index(drop=True)

# 5 Provinsi dengan harga turun/paling stabil = SURPLUS (Pemasok)
df_surplus = df.head(5).copy()
# 5 Provinsi dengan lonjakan harga tertinggi = DEFISIT (Penerima)
df_deficit = df.tail(5).copy()

# Simulasi Kapasitas Stok (Ton) - Dibuat seimbang agar logis
df_surplus['Supply'] = [1000, 800, 700, 600, 500] 
df_deficit['Demand'] = [900, 800, 700, 600, 600] 

# 7.3. Menghitung Matriks Jarak (Cost) menggunakan Geopy (As The Crow Flies)
biaya_jarak = {}
for i, row_s in df_surplus.iterrows():
    biaya_jarak[row_s['Province_Name']] = {}
    lokasi_surplus = (row_s['Latitude'], row_s['Longitude'])
    
    for j, row_d in df_deficit.iterrows():
        lokasi_defisit = (row_d['Latitude'], row_d['Longitude'])
        # Hitung jarak dalam Kilometer
        jarak_km = geodesic(lokasi_surplus, lokasi_defisit).kilometers
        biaya_jarak[row_s['Province_Name']][row_d['Province_Name']] = jarak_km

# 7.4. Inisialisasi Model Linear Programming (Minimisasi Cost)
model = pulp.LpProblem("Optimasi_Distribusi_Beras", pulp.LpMinimize)

# Membuat Variabel Keputusan (Berapa ton yang dikirim dari i ke j?)
rute = [(s, d) for s in df_surplus['Province_Name'] for d in df_deficit['Province_Name']]
x = pulp.LpVariable.dicts("Rute", (df_surplus['Province_Name'], df_deficit['Province_Name']), lowBound=0, cat='Continuous')

# 7.5. Fungsi Objektif: Minimalkan (Jarak * Volume)
model += pulp.lpSum([x[s][d] * biaya_jarak[s][d] for (s, d) in rute])

# 7.6. Kendala (Constraints)
# A. Kendala Suplai: Barang yang dikirim tidak boleh melebihi stok di gudang surplus
for s, stok in zip(df_surplus['Province_Name'], df_surplus['Supply']):
    model += pulp.lpSum([x[s][d] for d in df_deficit['Province_Name']]) <= stok

# B. Kendala Permintaan: Kebutuhan di provinsi defisit harus terpenuhi
for d, butuh in zip(df_deficit['Province_Name'], df_deficit['Demand']):
    model += pulp.lpSum([x[s][d] for s in df_surplus['Province_Name']]) == butuh

# 7.7. Selesaikan Model Optimasi
model.solve()

# 7.8. Tampilkan Hasil Rekomendasi Distribusi
print("="*50)
print("REKOMENDASI RUTE DISTRIBUSI TERENDAH BIAYA")
print("Status Optimasi:", pulp.LpStatus[model.status])
print("="*50)

for s in df_surplus['Province_Name']:
    for d in df_deficit['Province_Name']:
        if x[s][d].varValue > 0:
            jarak = biaya_jarak[s][d]
            print(f"Kirim {x[s][d].varValue} Ton dari [{s}] ---> ke [{d}] (Jarak: {jarak:.1f} KM)")

REKOMENDASI RUTE DISTRIBUSI TERENDAH BIAYA
Status Optimasi: Optimal
Kirim 200.0 Ton dari [Sumatera Barat] ---> ke [Jawa Tengah] (Jarak: 1280.1 KM)
Kirim 800.0 Ton dari [Sumatera Barat] ---> ke [Jawa Timur] (Jarak: 1534.9 KM)
Kirim 100.0 Ton dari [DKI Jakarta] ---> ke [Jawa Tengah] (Jarak: 372.7 KM)
Kirim 700.0 Ton dari [DKI Jakarta] ---> ke [DI Yogyakarta] (Jarak: 430.2 KM)
Kirim 600.0 Ton dari [Kepulauan Riau] ---> ke [Sulawesi Tengah] (Jarak: 1814.3 KM)
Kirim 100.0 Ton dari [Kepulauan Riau] ---> ke [Gorontalo] (Jarak: 1992.2 KM)
Kirim 600.0 Ton dari [Sumatera Selatan] ---> ke [Jawa Tengah] (Jarak: 767.3 KM)
Kirim 500.0 Ton dari [Papua] ---> ke [Gorontalo] (Jarak: 1774.9 KM)
